# Insurance Fraud Detection using ANN

Clean DL-engineer version of the project.

Main improvements added here:

- Proper EDA flow
- Duplicate, missing value, skewness and outlier checks
- Safer preprocessing using `ColumnTransformer`
- Ordinal encoding only for ordered columns
- One-hot encoding for nominal columns
- Train / validation / test split
- SMOTE only on training data
- ANN with Dropout, Batch Normalization, L2 and EarlyStopping
- Threshold tuning using validation data
- Final test evaluation only once
- Saved model, preprocessing pipeline, metadata and threshold

## 1. Import Libraries

In [ ]:
import os
import random
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["PYTHONHASHSEED"] = "42"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.metrics import AUC, Recall, Precision

import joblib

## 2. Reproducibility and Project Configuration

In [ ]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "data" / "raw" / "FraudDataset.csv").exists():
        PROJECT_ROOT = candidate
        break

DATA_PATH = str(PROJECT_ROOT / "data" / "raw" / "FraudDataset.csv")
TARGET = "FraudFound_P"

TEST_SIZE = 0.20
VAL_SIZE = 0.20
SMOTE_RATIO = 0.15

HIDDEN_LAYERS = [64, 32]
DROPOUT_RATE = 0.40
L2_LAMBDA = 0.01
LEARNING_RATE = 0.001
BATCH_SIZE = 32
EPOCHS = 100
PATIENCE = 15
CLASS_WEIGHT = {0: 1.0, 1: 5.0}

ARTIFACT_DIR = str(PROJECT_ROOT / "artifacts")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

## 3. Load Dataset

Keep the CSV file in the same folder as this notebook. If you are using Colab, upload the CSV file first.

In [ ]:
if not os.path.exists(DATA_PATH):
    print("File not found in current folder:", DATA_PATH)
    print("Place FraudDataset.csv inside data/raw or change DATA_PATH above.")
else:
    df_raw = pd.read_csv(DATA_PATH)
    print("Dataset loaded successfully")
    print("Rows   :", df_raw.shape[0])
    print("Columns:", df_raw.shape[1])

df_raw.head()

## 4. Basic Dataset Information

In [ ]:
print("Shape:", df_raw.shape)
print("\nColumns:")
print(df_raw.columns.tolist())
print("\nData types:")
print(df_raw.dtypes)

## 5. Missing Values and Duplicate Records

In [ ]:
missing_df = pd.DataFrame({
    "missing_count": df_raw.isnull().sum(),
    "missing_percent": (df_raw.isnull().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)

print("Total duplicate rows:", df_raw.duplicated().sum())
print("\nMissing value summary:")
display(missing_df[missing_df["missing_count"] > 0])

if missing_df["missing_count"].sum() == 0:
    print("No missing values found")

## 6. Target Distribution

In [ ]:
target_counts = df_raw[TARGET].value_counts().sort_index()
target_percent = (df_raw[TARGET].value_counts(normalize=True).sort_index() * 100).round(2)

target_summary = pd.DataFrame({
    "count": target_counts,
    "percent": target_percent
})

display(target_summary)

plt.figure(figsize=(5, 4))
sns.countplot(x=TARGET, data=df_raw)
plt.title("Fraud vs Non-Fraud Distribution")
plt.xlabel("0 = Non-Fraud, 1 = Fraud")
plt.ylabel("Count")
plt.show()

print("Fraud percentage:", round(df_raw[TARGET].mean() * 100, 2), "%")

## 7. Numerical and Categorical Columns

In [ ]:
num_cols_raw = df_raw.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols_raw = df_raw.select_dtypes(include="object").columns.tolist()

print("Numerical columns:", len(num_cols_raw))
print(num_cols_raw)
print("\nCategorical columns:", len(cat_cols_raw))
print(cat_cols_raw)

## 8. Numerical Summary and Skewness

In [ ]:
display(df_raw[num_cols_raw].describe().T)

skew_df = pd.DataFrame({
    "skewness": df_raw[num_cols_raw].skew().round(3)
}).sort_values("skewness", ascending=False)

display(skew_df)

## 9. IQR Outlier Check

Outliers are checked for understanding only. In fraud detection, unusual values may be useful signals, so they should not be removed blindly.

In [ ]:
outlier_rows = []

for col in num_cols_raw:
    if col == TARGET:
        continue
    q1 = df_raw[col].quantile(0.25)
    q3 = df_raw[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_count = ((df_raw[col] < lower_bound) | (df_raw[col] > upper_bound)).sum()
    outlier_rows.append([col, iqr, lower_bound, upper_bound, outlier_count])

outlier_df = pd.DataFrame(outlier_rows, columns=["column", "iqr", "lower_bound", "upper_bound", "outlier_count"])
outlier_df = outlier_df.sort_values("outlier_count", ascending=False)
display(outlier_df)

## 10. Important Boxplots

In [ ]:
plot_cols = [col for col in ["Age", "Deductible", "DriverRating", "WeekOfMonth", "WeekOfMonthClaimed"] if col in df_raw.columns]

for col in plot_cols:
    plt.figure(figsize=(6, 3))
    sns.boxplot(x=df_raw[col])
    plt.title("Boxplot of " + col)
    plt.show()

## 11. Fraud Rate by Important Categorical Features

In [ ]:
important_cat_cols = [
    "Fault",
    "PolicyType",
    "VehicleCategory",
    "BasePolicy",
    "VehiclePrice",
    "AgeOfVehicle",
    "PastNumberOfClaims",
    "AddressChange_Claim"
]

for col in important_cat_cols:
    if col in df_raw.columns:
        fraud_rate = df_raw.groupby(col)[TARGET].agg(["count", "sum", "mean"]).sort_values("mean", ascending=False)
        fraud_rate["fraud_rate_percent"] = (fraud_rate["mean"] * 100).round(2)
        print("\n", col)
        display(fraud_rate[["count", "sum", "fraud_rate_percent"]])

## 12. Correlation Matrix for EDA

Categorical columns are factorized only for correlation visualization. This is not used for model training.

In [ ]:
df_corr = df_raw.copy()

for col in df_corr.select_dtypes(include="object").columns:
    df_corr[col] = pd.factorize(df_corr[col])[0]

corr_target = df_corr.corr(numeric_only=True)[TARGET].sort_values(ascending=False)
display(corr_target)

plt.figure(figsize=(14, 10))
sns.heatmap(df_corr.corr(numeric_only=True), cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.show()

## 13. Data Cleaning

In [ ]:
df = df_raw.copy()

drop_cols = ["PolicyNumber", "RepNumber", "Year"]
drop_cols = [col for col in drop_cols if col in df.columns]
df = df.drop(columns=drop_cols)

if "Age" in df.columns:
    age_median = df.loc[df["Age"] > 0, "Age"].median()
    df["Age"] = df["Age"].replace(0, age_median)

for col in ["DayOfWeekClaimed", "MonthClaimed"]:
    if col in df.columns:
        valid_mode = df.loc[df[col].astype(str) != "0", col].mode()[0]
        df[col] = df[col].replace("0", valid_mode)

print("Dropped columns:", drop_cols)
print("Cleaned shape:", df.shape)
print("Age zero count:", (df["Age"] == 0).sum())
print("DayOfWeekClaimed zero count:", (df["DayOfWeekClaimed"].astype(str) == "0").sum())
print("MonthClaimed zero count:", (df["MonthClaimed"].astype(str) == "0").sum())

## 14. Define Feature Groups

Ordinal columns have a natural order. Nominal columns do not have order, so one-hot encoding is used for them.

In [ ]:
ordinal_map = {
    "VehiclePrice": ["less than 20000", "20000 to 29000", "30000 to 39000", "40000 to 59000", "60000 to 69000", "more than 69000"],
    "Days_Policy_Accident": ["none", "1 to 7", "8 to 15", "15 to 30", "more than 30"],
    "Days_Policy_Claim": ["none", "8 to 15", "15 to 30", "more than 30"],
    "PastNumberOfClaims": ["none", "1", "2 to 4", "more than 4"],
    "AgeOfVehicle": ["new", "2 years", "3 years", "4 years", "5 years", "6 years", "7 years", "more than 7"],
    "AgeOfPolicyHolder": ["16 to 17", "18 to 20", "21 to 25", "26 to 30", "31 to 35", "36 to 40", "41 to 50", "51 to 65", "over 65"],
    "NumberOfSuppliments": ["none", "1 to 2", "3 to 5", "more than 5"],
    "AddressChange_Claim": ["no change", "under 6 months", "1 year", "2 to 3 years", "4 to 8 years"],
    "NumberOfCars": ["1 vehicle", "2 vehicles", "3 to 4", "5 to 8", "more than 8"]
}

X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

ordinal_cols = [col for col in ordinal_map.keys() if col in X.columns]
ordinal_categories = [ordinal_map[col] for col in ordinal_cols]

numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
nominal_cols = [col for col in X.select_dtypes(include="object").columns.tolist() if col not in ordinal_cols]

print("Numeric columns:", numeric_cols)
print("Ordinal columns:", ordinal_cols)
print("Nominal columns:", nominal_cols)
print("Target distribution:")
print(y.value_counts(normalize=True).round(4))

## 15. Train, Validation and Test Split

The test set is kept untouched until final evaluation.

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=VAL_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_train_val
)

print("Train shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_val.shape, y_val.shape)
print("Test shape:", X_test.shape, y_test.shape)
print("\nTrain fraud rate:", round(y_train.mean() * 100, 2), "%")
print("Validation fraud rate:", round(y_val.mean() * 100, 2), "%")
print("Test fraud rate:", round(y_test.mean() * 100, 2), "%")

## 16. Preprocessing Pipeline

This avoids the main previous issue: nominal columns are no longer label encoded.

In [ ]:
try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

ordinal_pipe = Pipeline(steps=[
    ("ordinal", OrdinalEncoder(
        categories=ordinal_categories,
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("ord", ordinal_pipe, ordinal_cols),
        ("nom", onehot, nominal_cols)
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

print("Processed train shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Processed test shape:", X_test_processed.shape)
print("Total final features:", len(feature_names))

## 17. Apply SMOTE on Training Data Only

In [ ]:
smote = SMOTE(
    sampling_strategy=SMOTE_RATIO,
    random_state=RANDOM_STATE,
    k_neighbors=5
)

X_train_smote, y_train_smote = smote.fit_resample(X_train_processed, y_train)

print("Before SMOTE:")
print(y_train.value_counts())
print("\nAfter SMOTE:")
print(pd.Series(y_train_smote).value_counts())
print("\nValidation and test data are not touched by SMOTE")

## 18. Baseline ANN Model

In [ ]:
tf.keras.backend.clear_session()

baseline = Sequential([
    Input(shape=(X_train_processed.shape[1],), name="input"),
    Dense(64, activation="relu", name="dense_1"),
    Dense(1, activation="sigmoid", name="output")
])

baseline.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=[AUC(name="auc"), Recall(name="recall"), Precision(name="precision")]
)

baseline.summary()

## 19. Train Baseline ANN

In [ ]:
es_baseline = EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=8,
    restore_best_weights=True,
    verbose=1
)

history_baseline = baseline.fit(
    X_train_smote,
    y_train_smote,
    validation_data=(X_val_processed, y_val),
    epochs=50,
    batch_size=BATCH_SIZE,
    class_weight=CLASS_WEIGHT,
    callbacks=[es_baseline],
    verbose=1
)

## 20. Final Regularized ANN Model

In [ ]:
def build_ann(input_dim):
    tf.keras.backend.clear_session()
    model = Sequential()
    model.add(Input(shape=(input_dim,), name="input"))

    for i, units in enumerate(HIDDEN_LAYERS):
        model.add(Dense(
            units,
            activation="relu",
            kernel_regularizer=l2(L2_LAMBDA),
            name="dense_" + str(i + 1)
        ))
        model.add(BatchNormalization(name="bn_" + str(i + 1)))
        model.add(Dropout(DROPOUT_RATE, name="dropout_" + str(i + 1)))

    model.add(Dense(1, activation="sigmoid", name="output"))

    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=[AUC(name="auc"), Recall(name="recall"), Precision(name="precision")]
    )
    return model

ann = build_ann(X_train_processed.shape[1])
ann.summary()

## 21. Train Final ANN Model

In [ ]:
checkpoint_path = os.path.join(ARTIFACT_DIR, "best_ann_model.keras")

callbacks = [
    EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=checkpoint_path,
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    )
]

history = ann.fit(
    X_train_smote,
    y_train_smote,
    validation_data=(X_val_processed, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=CLASS_WEIGHT,
    callbacks=callbacks,
    verbose=1
)

best_epoch = np.argmax(history.history["val_auc"]) + 1
best_auc = max(history.history["val_auc"])
best_recall = history.history["val_recall"][best_epoch - 1]

print("Training complete")
print("Total epochs ran:", len(history.history["loss"]))
print("Best epoch:", best_epoch)
print("Best validation AUC:", round(best_auc, 4))
print("Validation recall at best AUC epoch:", round(best_recall, 4))

## 22. Training Curves

In [ ]:
history_df = pd.DataFrame(history.history)
display(history_df.tail())

plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="train_loss")
plt.plot(history.history["val_loss"], label="val_loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history.history["auc"], label="train_auc")
plt.plot(history.history["val_auc"], label="val_auc")
plt.title("Training and Validation AUC")
plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history.history["recall"], label="train_recall")
plt.plot(history.history["val_recall"], label="val_recall")
plt.title("Training and Validation Recall")
plt.xlabel("Epoch")
plt.ylabel("Recall")
plt.legend()
plt.show()

## 23. Threshold Tuning on Validation Data

Threshold is selected on validation data, not on test data.

In [ ]:
def get_metric_row(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

val_prob = ann.predict(X_val_processed).ravel()
thresholds = np.arange(0.10, 0.91, 0.05)
threshold_results = pd.DataFrame([get_metric_row(y_val, val_prob, t) for t in thresholds])

best_threshold = threshold_results.sort_values(["f2", "recall"], ascending=False).iloc[0]["threshold"]

print("Best threshold based on validation F2:", round(best_threshold, 2))
display(threshold_results.sort_values("f2", ascending=False).head(10))

## 24. Final Test Evaluation

In [ ]:
test_prob = ann.predict(X_test_processed).ravel()

test_metrics_05 = get_metric_row(y_test, test_prob, 0.50)
test_metrics_best = get_metric_row(y_test, test_prob, best_threshold)

comparison_df = pd.DataFrame([test_metrics_05, test_metrics_best])
comparison_df["roc_auc"] = roc_auc_score(y_test, test_prob)
comparison_df["pr_auc"] = average_precision_score(y_test, test_prob)

display(comparison_df)

final_pred = (test_prob >= best_threshold).astype(int)
print("Classification Report at selected threshold:")
print(classification_report(y_test, final_pred, target_names=["Non-Fraud", "Fraud"], zero_division=0))

## 25. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, final_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Non-Fraud", "Fraud"], yticklabels=["Non-Fraud", "Fraud"])
plt.title("Final ANN Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

tn, fp, fn, tp = cm.ravel()
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)
print("\nBusiness meaning:")
print("False Negative means fraud missed, which is costly for the insurance company.")
print("False Positive means genuine claim sent for manual review.")

## 26. ROC Curve and Precision-Recall Curve

In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_test, test_prob)
precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_test, test_prob)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label="ROC-AUC = " + str(round(roc_auc_score(y_test, test_prob), 4)))
plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(recall_curve, precision_curve, label="PR-AUC = " + str(round(average_precision_score(y_test, test_prob), 4)))
plt.title("Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.show()

## 27. Classical ML Model Comparison

These models are trained on the same preprocessed data for a fair baseline comparison.

In [ ]:
ml_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=8, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE)
}

ml_results = []

for name, model in ml_models.items():
    model.fit(X_train_processed, y_train)
    if hasattr(model, "predict_proba"):
        prob = model.predict_proba(X_test_processed)[:, 1]
    else:
        prob = model.decision_function(X_test_processed)
    pred = (prob >= 0.50).astype(int)
    ml_results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "f2": fbeta_score(y_test, pred, beta=2, zero_division=0),
        "roc_auc": roc_auc_score(y_test, prob),
        "pr_auc": average_precision_score(y_test, prob)
    })

ann_result = {
    "model": "Final ANN",
    "accuracy": accuracy_score(y_test, final_pred),
    "precision": precision_score(y_test, final_pred, zero_division=0),
    "recall": recall_score(y_test, final_pred, zero_division=0),
    "f1": f1_score(y_test, final_pred, zero_division=0),
    "f2": fbeta_score(y_test, final_pred, beta=2, zero_division=0),
    "roc_auc": roc_auc_score(y_test, test_prob),
    "pr_auc": average_precision_score(y_test, test_prob)
}

results_df = pd.DataFrame(ml_results + [ann_result]).sort_values("recall", ascending=False)
display(results_df)

## 28. Feature Importance from Random Forest

This gives a basic explainability view. For production, SHAP can be added later.

In [ ]:
rf_model = ml_models["Random Forest"]
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False).head(20)

display(importance_df)

plt.figure(figsize=(8, 6))
sns.barplot(data=importance_df, x="importance", y="feature")
plt.title("Top 20 Feature Importances - Random Forest")
plt.show()

## 29. Save Model and Complete Preprocessing Pipeline

In [ ]:
final_model_path = os.path.join(ARTIFACT_DIR, "fraud_ann_model.keras")
preprocessor_path = os.path.join(ARTIFACT_DIR, "preprocessor.joblib")
metadata_path = os.path.join(ARTIFACT_DIR, "project_metadata.joblib")

ann.save(final_model_path)
joblib.dump(preprocessor, preprocessor_path)

metadata = {
    "target": TARGET,
    "drop_cols": drop_cols,
    "numeric_cols": numeric_cols,
    "ordinal_cols": ordinal_cols,
    "nominal_cols": nominal_cols,
    "ordinal_map": ordinal_map,
    "feature_names": list(feature_names),
    "threshold": float(best_threshold),
    "smote_ratio": SMOTE_RATIO,
    "class_weight": CLASS_WEIGHT
}

joblib.dump(metadata, metadata_path)

print("Saved model:", final_model_path)
print("Saved preprocessor:", preprocessor_path)
print("Saved metadata:", metadata_path)

## 30. Inference Test on One Sample

In [ ]:
loaded_model = load_model(final_model_path)
loaded_preprocessor = joblib.load(preprocessor_path)
loaded_metadata = joblib.load(metadata_path)

sample = X_test.iloc[[0]].copy()
sample_processed = loaded_preprocessor.transform(sample)
sample_prob = loaded_model.predict(sample_processed).ravel()[0]
sample_pred = int(sample_prob >= loaded_metadata["threshold"])

print("Fraud probability:", round(sample_prob, 4))
print("Selected threshold:", round(loaded_metadata["threshold"], 2))
print("Prediction:", sample_pred)

if sample_pred == 1:
    print("Decision: Send claim for fraud review")
else:
    print("Decision: Claim looks non-fraud based on model threshold")

## 31. Final Viva Summary

In [ ]:
print("Project Type: Binary Classification")
print("Business Goal: Detect fraudulent insurance claims")
print("Why accuracy is not enough: Dataset is highly imbalanced")
print("Most important metric: Fraud recall and F2 score")
print("Main preprocessing improvement: One-hot encoding for nominal columns and ordinal encoding for ordered columns")
print("Leakage control: Preprocessor fitted on training data only and SMOTE applied only on training data")
print("Final model: ANN with ReLU, BatchNorm, Dropout, L2, Adam, Binary Crossentropy")
print("Deployment artifacts: ANN model, preprocessor, metadata and threshold")